# 🧠 PPO: Complete Research Notebook
### *For the Advanced Researcher / PhD Student*

---

## What You Will Learn

This notebook is structured as a **progressive research investigation** of PPO:

| Section | Focus |
|---------|-------|
| 1 | Environment Setup & Baselines |
| 2 | PPO from Scratch — Every component exposed |
| 3 | GAE Deep Dive — λ tradeoffs |
| 4 | Clipping ε tradeoffs |
| 5 | Value Function — architecture & loss tradeoffs |
| 6 | Entropy coefficient tradeoffs |
| 7 | Epoch & Batch size tradeoffs |
| 8 | Network architecture tradeoffs |
| 9 | KL Divergence monitoring & early stopping |
| 10 | Full ablation study across CartPole & LunarLander |
| 11 | Diagnostic tools: gradient norms, explained variance |
| 12 | PPO-KL vs PPO-Clip comparison |

---

> **Philosophy:** We don't just train — we **instrument**, **ablate**, and **understand** every decision.


## Section 0: Installation & Imports

In [ ]:
# Install dependencies
!pip install gymnasium[box2d] gymnasium[classic-control] -q
!pip install torch torchvision -q
!apt-get install -y xvfb -q  # virtual display for rendering
!pip install pyvirtualdisplay -q

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical, Normal
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import defaultdict, deque
import time
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')
print(f'Gymnasium version: {gym.__version__}')

In [ ]:
# ── Plotting utilities ──────────────────────────────────────────────────────

def smooth(arr, window=20):
    """Exponential moving average smoothing."""
    smoothed = []
    ema = arr[0]
    for x in arr:
        ema = 0.9 * ema + 0.1 * x
        smoothed.append(ema)
    return smoothed

def plot_training(logs, title='PPO Training', figsize=(16, 10)):
    """Comprehensive training dashboard."""
    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    # Episode rewards
    ax1 = fig.add_subplot(gs[0, :])
    ep_rewards = logs['episode_rewards']
    ax1.plot(ep_rewards, alpha=0.3, color='steelblue', label='Raw')
    ax1.plot(smooth(ep_rewards), color='steelblue', linewidth=2, label='EMA')
    ax1.set_title('Episode Return', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Episode'); ax1.set_ylabel('Return')
    ax1.legend(); ax1.grid(alpha=0.3)

    # Policy loss
    ax2 = fig.add_subplot(gs[1, 0])
    ax2.plot(logs['policy_loss'], color='tomato')
    ax2.set_title('Policy Loss', fontsize=11); ax2.grid(alpha=0.3)

    # Value loss
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.plot(logs['value_loss'], color='orange')
    ax3.set_title('Value Loss', fontsize=11); ax3.grid(alpha=0.3)

    # Entropy
    ax4 = fig.add_subplot(gs[1, 2])
    ax4.plot(logs['entropy'], color='mediumseagreen')
    ax4.set_title('Policy Entropy', fontsize=11); ax4.grid(alpha=0.3)

    # KL divergence
    ax5 = fig.add_subplot(gs[2, 0])
    ax5.plot(logs['approx_kl'], color='purple')
    ax5.axhline(0.02, color='red', linestyle='--', alpha=0.5, label='Target KL')
    ax5.set_title('Approx KL Divergence', fontsize=11); ax5.legend(); ax5.grid(alpha=0.3)

    # Clip fraction
    ax6 = fig.add_subplot(gs[2, 1])
    ax6.plot(logs['clip_fraction'], color='goldenrod')
    ax6.set_title('Clip Fraction', fontsize=11); ax6.grid(alpha=0.3)

    # Explained variance
    ax7 = fig.add_subplot(gs[2, 2])
    ax7.plot(logs['explained_variance'], color='teal')
    ax7.axhline(0, color='black', linestyle='--', alpha=0.3)
    ax7.axhline(1, color='green', linestyle='--', alpha=0.3, label='Perfect')
    ax7.set_title('Explained Variance (Critic)', fontsize=11); ax7.legend(); ax7.grid(alpha=0.3)

    fig.suptitle(title, fontsize=15, fontweight='bold', y=1.01)
    plt.show()

print('Plotting utilities loaded ✓')

---
## Section 1: Environments & Baselines

We use two environments with increasing difficulty:

| Environment | Obs Space | Action Space | Max Return | Notes |
|------------|-----------|--------------|------------|-------|
| CartPole-v1 | 4-dim continuous | 2 discrete | 500 | Simple, fast to train |
| LunarLander-v2 | 8-dim continuous | 4 discrete | ~250 | Harder, good for ablations |

In [ ]:
# ── Environment wrappers ────────────────────────────────────────────────────

def make_env(env_id, seed=0):
    """Create and seed an environment."""
    env = gym.make(env_id)
    env = gym.wrappers.RecordEpisodeStatistics(env)  # auto-tracks returns/lengths
    env.reset(seed=seed)
    return env

# Inspect environments
for env_id in ['CartPole-v1', 'LunarLander-v2']:
    env = make_env(env_id)
    obs, _ = env.reset()
    print(f'\n{env_id}:')
    print(f'  Observation shape : {env.observation_space.shape}')
    print(f'  Action space      : {env.action_space}')
    print(f'  Reward range      : {env.reward_range}')
    env.close()

# Random baseline (what does a random agent achieve?)
def random_baseline(env_id, n_episodes=100):
    env = make_env(env_id)
    returns = []
    for _ in range(n_episodes):
        obs, _ = env.reset()
        done = False
        total_r = 0
        while not done:
            action = env.action_space.sample()
            obs, r, term, trunc, _ = env.step(action)
            total_r += r
            done = term or trunc
        returns.append(total_r)
    env.close()
    return np.mean(returns), np.std(returns)

for env_id in ['CartPole-v1', 'LunarLander-v2']:
    mean, std = random_baseline(env_id)
    print(f'{env_id} random baseline: {mean:.1f} ± {std:.1f}')

---
## Section 2: PPO from Scratch — Every Component Exposed

We implement PPO with **maximum transparency** — every component is separate, logged, and inspectable.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2.1  ACTOR-CRITIC NETWORK
# ══════════════════════════════════════════════════════════════════════════════

def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    """
    Orthogonal initialization — standard PPO practice.
    std=sqrt(2) for hidden layers, std=0.01 for policy head (near-uniform init),
    std=1.0 for value head.
    This prevents early bias toward any action and keeps value estimates small.
    """
    nn.init.orthogonal_(layer.weight, std)
    nn.init.constant_(layer.bias, bias_const)
    return layer


class ActorCritic(nn.Module):
    """
    Shared-trunk Actor-Critic.
    
    Architecture:
        Shared: obs → [64 → 64]  (feature extractor)
        Actor head:  features → logits → Categorical
        Critic head: features → scalar V(s)
    
    Research note: sharing vs. separate networks is a key hyperparameter.
    Shared = faster but can cause gradient interference.
    Separate = more stable but 2x parameters.
    """
    def __init__(self, obs_dim, n_actions, hidden_size=64):
        super().__init__()
        self.shared = nn.Sequential(
            layer_init(nn.Linear(obs_dim, hidden_size)),
            nn.Tanh(),
            layer_init(nn.Linear(hidden_size, hidden_size)),
            nn.Tanh(),
        )
        # Policy head: small std → near-uniform initial policy
        self.actor = layer_init(nn.Linear(hidden_size, n_actions), std=0.01)
        # Value head: std=1.0 for reasonable initial value estimates
        self.critic = layer_init(nn.Linear(hidden_size, 1), std=1.0)

    def get_value(self, x):
        return self.critic(self.shared(x))

    def get_action_and_value(self, x, action=None):
        features = self.shared(x)
        logits = self.actor(features)
        dist = Categorical(logits=logits)
        if action is None:
            action = dist.sample()
        return action, dist.log_prob(action), dist.entropy(), self.critic(features)

print('ActorCritic network defined ✓')

# Quick test
net = ActorCritic(obs_dim=4, n_actions=2)
dummy = torch.randn(8, 4)
a, lp, ent, v = net.get_action_and_value(dummy)
print(f'  action shape: {a.shape}, log_prob: {lp.shape}, value: {v.shape}')
print(f'  param count: {sum(p.numel() for p in net.parameters()):,}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2.2  ROLLOUT BUFFER
# ══════════════════════════════════════════════════════════════════════════════

class RolloutBuffer:
    """
    Stores T timesteps of experience.
    
    Key design choices:
    - Pre-allocate arrays for speed
    - Store log_probs from OLD policy (needed for importance sampling ratio)
    - Compute GAE advantages after buffer is full
    """
    def __init__(self, T, obs_dim, device):
        self.T = T
        self.device = device
        self.obs = torch.zeros((T, obs_dim), device=device)
        self.actions = torch.zeros(T, dtype=torch.long, device=device)
        self.rewards = torch.zeros(T, device=device)
        self.values = torch.zeros(T, device=device)
        self.log_probs = torch.zeros(T, device=device)
        self.dones = torch.zeros(T, device=device)
        self.ptr = 0

    def store(self, obs, action, reward, value, log_prob, done):
        self.obs[self.ptr] = obs
        self.actions[self.ptr] = action
        self.rewards[self.ptr] = reward
        self.values[self.ptr] = value
        self.log_probs[self.ptr] = log_prob
        self.dones[self.ptr] = done
        self.ptr += 1

    def compute_gae(self, last_value, gamma, lam):
        """
        Compute Generalized Advantage Estimation.
        
        GAE formula (backward pass):
            delta_t = r_t + gamma * V(s_{t+1}) * (1 - done) - V(s_t)
            A_t = delta_t + gamma * lam * (1 - done) * A_{t+1}

        The (1 - done) masks ensure we don't bootstrap
        across episode boundaries.
        """
        advantages = torch.zeros_like(self.rewards)
        last_gae = 0.0

        for t in reversed(range(self.T)):
            if t == self.T - 1:
                next_non_terminal = 1.0 - self.dones[t]  # 0 if done, 1 if not
                next_value = last_value
            else:
                next_non_terminal = 1.0 - self.dones[t]
                next_value = self.values[t + 1]

            # TD error (delta)
            delta = self.rewards[t] + gamma * next_value * next_non_terminal - self.values[t]

            # Accumulate GAE
            last_gae = delta + gamma * lam * next_non_terminal * last_gae
            advantages[t] = last_gae

        # Returns = advantages + values (used as critic targets)
        returns = advantages + self.values
        return advantages, returns

    def get_batches(self, batch_size):
        """Random mini-batches."""
        idx = torch.randperm(self.T, device=self.device)
        for start in range(0, self.T, batch_size):
            batch_idx = idx[start:start + batch_size]
            yield (self.obs[batch_idx], self.actions[batch_idx],
                   self.log_probs[batch_idx], self.advantages[batch_idx],
                   self.returns[batch_idx])

    def prepare(self, last_value, gamma, lam):
        """Compute advantages, normalize, store."""
        adv, ret = self.compute_gae(last_value, gamma, lam)
        # Normalize advantages: zero mean, unit std
        self.advantages = (adv - adv.mean()) / (adv.std() + 1e-8)
        self.returns = ret
        self.ptr = 0  # reset pointer

print('RolloutBuffer defined ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2.3  PPO DIAGNOSTICS — The Research Layer
# ══════════════════════════════════════════════════════════════════════════════

def explained_variance(y_pred, y_true):
    """
    Fraction of variance in returns explained by value function.
    
    Interpretation:
        ~1.0 : critic perfectly predicts returns (ideal)
         0.0 : critic no better than mean prediction
        <0.0 : critic is actively misleading (bad!)
    
    Key metric for diagnosing value function quality.
    If EV is low, GAE advantages are noisy → policy gradient is noisy.
    """
    y_pred = y_pred.detach().cpu().numpy()
    y_true = y_true.detach().cpu().numpy()
    var_y = np.var(y_true)
    return np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y


def compute_approx_kl(old_log_probs, new_log_probs):
    """
    Approximate KL divergence: E[log(pi_old/pi_new)].
    
    This is the reverse KL used in PPO monitoring.
    Rule of thumb: KL > 0.02 suggests policy is changing too fast.
    John Schulman recommends stopping epochs if KL > 1.5 * target_kl.
    """
    log_ratio = old_log_probs - new_log_probs
    return torch.mean(log_ratio).item()


def compute_clip_fraction(ratio, clip_eps):
    """
    Fraction of samples where clipping was active.
    
    Interpretation:
        ~0.0 : policy barely changing (might be too conservative)
        ~0.1-0.3: healthy clipping
        ~0.5+: policy changing too much, reduce lr or increase clip interval
    """
    clipped = (ratio - 1.0).abs() > clip_eps
    return clipped.float().mean().item()


def compute_grad_norm(model):
    """L2 norm of gradients — detects exploding/vanishing gradients."""
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            total_norm += p.grad.data.norm(2).item() ** 2
    return total_norm ** 0.5


print('Diagnostic functions defined ✓')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2.4  THE PPO TRAINER — Full Implementation
# ══════════════════════════════════════════════════════════════════════════════

class PPOConfig:
    """
    All hyperparameters in one place.
    Default values are from the original PPO paper (Schulman et al., 2017).
    """
    # Environment
    env_id: str = 'CartPole-v1'

    # Rollout
    total_timesteps: int = 500_000
    rollout_steps: int = 2048   # T — steps per rollout

    # GAE
    gamma: float = 0.99         # discount factor
    gae_lambda: float = 0.95    # GAE lambda (bias-variance tradeoff)

    # PPO update
    n_epochs: int = 10          # K — epochs per update
    batch_size: int = 64        # mini-batch size
    clip_eps: float = 0.2       # ε for clipping
    target_kl: float = None     # early stopping KL threshold (None = disabled)

    # Loss coefficients
    vf_coef: float = 0.5        # c1 — value loss weight
    ent_coef: float = 0.01      # c2 — entropy bonus weight

    # Optimization
    lr: float = 3e-4
    max_grad_norm: float = 0.5  # gradient clipping
    anneal_lr: bool = True      # linearly anneal LR to 0

    # Network
    hidden_size: int = 64

    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)


class PPOTrainer:
    """
    Full PPO implementation with rich diagnostics.
    """
    def __init__(self, config: PPOConfig):
        self.cfg = config
        self.env = make_env(config.env_id)
        obs_dim = self.env.observation_space.shape[0]
        n_actions = self.env.action_space.n

        self.agent = ActorCritic(obs_dim, n_actions, config.hidden_size).to(device)
        self.optimizer = optim.Adam(self.agent.parameters(), lr=config.lr, eps=1e-5)

        self.buffer = RolloutBuffer(config.rollout_steps, obs_dim, device)

        # Logging
        self.logs = defaultdict(list)
        self.global_step = 0

    # ── Phase 1: Collect Rollouts ──────────────────────────────────────────
    def collect_rollout(self, obs):
        obs_tensor = torch.FloatTensor(obs).to(device)

        for t in range(self.cfg.rollout_steps):
            self.global_step += 1

            with torch.no_grad():
                action, log_prob, _, value = self.agent.get_action_and_value(obs_tensor)

            next_obs, reward, term, trunc, info = self.env.step(action.item())
            done = term or trunc

            self.buffer.store(
                obs_tensor, action, torch.tensor(reward),
                value.squeeze(), log_prob, torch.tensor(float(done))
            )

            # Log completed episodes
            if done and 'episode' in info:
                self.logs['episode_rewards'].append(info['episode']['r'])
                self.logs['episode_lengths'].append(info['episode']['l'])

            obs = next_obs if not done else self.env.reset()[0]
            obs_tensor = torch.FloatTensor(obs).to(device)

        # Bootstrap: get value of the state we stopped at
        with torch.no_grad():
            last_value = self.agent.get_value(obs_tensor).squeeze()

        self.buffer.prepare(last_value, self.cfg.gamma, self.cfg.gae_lambda)
        return obs

    # ── Phase 2: PPO Update ────────────────────────────────────────────────
    def update(self, iteration, total_iters):
        # Learning rate annealing: linearly decay to 0
        if self.cfg.anneal_lr:
            frac = 1.0 - (iteration / total_iters)
            self.optimizer.param_groups[0]['lr'] = frac * self.cfg.lr

        epoch_metrics = defaultdict(list)

        for epoch in range(self.cfg.n_epochs):
            for obs_b, act_b, old_lp_b, adv_b, ret_b in \
                    self.buffer.get_batches(self.cfg.batch_size):

                # Evaluate current policy on old data
                _, new_lp, entropy, new_val = self.agent.get_action_and_value(obs_b, act_b)
                new_val = new_val.squeeze()

                # ── Importance sampling ratio ──
                log_ratio = new_lp - old_lp_b
                ratio = log_ratio.exp()   # r_t(θ) = π_new / π_old

                # ── Policy loss (PPO-Clip) ──
                surr1 = ratio * adv_b
                surr2 = torch.clamp(ratio, 1 - self.cfg.clip_eps,
                                    1 + self.cfg.clip_eps) * adv_b
                policy_loss = -torch.min(surr1, surr2).mean()

                # ── Value loss ──
                # Option A: plain MSE
                value_loss = 0.5 * ((new_val - ret_b) ** 2).mean()

                # ── Entropy bonus ──
                entropy_loss = entropy.mean()

                # ── Combined loss ──
                loss = (policy_loss
                        + self.cfg.vf_coef * value_loss
                        - self.cfg.ent_coef * entropy_loss)

                # ── Gradient update ──
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.agent.parameters(),
                                         self.cfg.max_grad_norm)
                self.optimizer.step()

                # Diagnostics
                with torch.no_grad():
                    approx_kl = compute_approx_kl(old_lp_b, new_lp)
                    clip_frac = compute_clip_fraction(ratio, self.cfg.clip_eps)
                    grad_norm = compute_grad_norm(self.agent)
                    ev = explained_variance(new_val, ret_b)

                epoch_metrics['policy_loss'].append(policy_loss.item())
                epoch_metrics['value_loss'].append(value_loss.item())
                epoch_metrics['entropy'].append(entropy_loss.item())
                epoch_metrics['approx_kl'].append(approx_kl)
                epoch_metrics['clip_fraction'].append(clip_frac)
                epoch_metrics['grad_norm'].append(grad_norm)
                epoch_metrics['explained_variance'].append(ev)

            # Early stopping on KL
            if self.cfg.target_kl is not None:
                mean_kl = np.mean(epoch_metrics['approx_kl'])
                if mean_kl > 1.5 * self.cfg.target_kl:
                    break

        # Aggregate metrics across epochs
        for k, v in epoch_metrics.items():
            self.logs[k].append(np.mean(v))

    # ── Main training loop ─────────────────────────────────────────────────
    def train(self, verbose=True):
        obs, _ = self.env.reset(seed=SEED)
        total_iters = self.cfg.total_timesteps // self.cfg.rollout_steps

        t_start = time.time()
        for iteration in range(total_iters):
            obs = self.collect_rollout(obs)
            self.update(iteration, total_iters)

            if verbose and iteration % 10 == 0 and self.logs['episode_rewards']:
                recent = np.mean(self.logs['episode_rewards'][-20:])
                ev = self.logs['explained_variance'][-1] if self.logs['explained_variance'] else 0
                kl = self.logs['approx_kl'][-1] if self.logs['approx_kl'] else 0
                elapsed = time.time() - t_start
                sps = self.global_step / elapsed
                print(f'  iter {iteration:4d}/{total_iters} | '
                      f'steps {self.global_step:7d} | '
                      f'return {recent:7.1f} | '
                      f'EV {ev:.3f} | '
                      f'KL {kl:.4f} | '
                      f'{sps:.0f} sps')

        self.env.close()
        print(f'\nTraining complete. Total time: {time.time()-t_start:.1f}s')
        return self.logs


print('PPOTrainer defined ✓')

---
## Section 3: Baseline Run on CartPole

In [ ]:
print('=' * 60)
print('BASELINE PPO — CartPole-v1')
print('=' * 60)

cfg_baseline = PPOConfig(
    env_id='CartPole-v1',
    total_timesteps=300_000,
    rollout_steps=2048,
    gamma=0.99,
    gae_lambda=0.95,
    n_epochs=10,
    batch_size=64,
    clip_eps=0.2,
    vf_coef=0.5,
    ent_coef=0.01,
    lr=3e-4,
    max_grad_norm=0.5,
    hidden_size=64
)

trainer_baseline = PPOTrainer(cfg_baseline)
logs_baseline = trainer_baseline.train(verbose=True)

plot_training(logs_baseline, title='Baseline PPO — CartPole-v1')

In [ ]:
# Diagnostic summary
print('\n── DIAGNOSTIC SUMMARY ──────────────────────────────────────────')

def summarize_logs(logs, label=''):
    if logs['episode_rewards']:
        print(f'\n{label}')
        print(f'  Final return (last 50 eps)  : {np.mean(logs["episode_rewards"][-50:]):.1f}')
        print(f'  Peak return                 : {max(logs["episode_rewards"]):.1f}')
        print(f'  Final explained variance    : {np.mean(logs["explained_variance"][-10:]):.3f}')
        print(f'  Final approx KL             : {np.mean(logs["approx_kl"][-10:]):.4f}')
        print(f'  Final clip fraction         : {np.mean(logs["clip_fraction"][-10:]):.3f}')
        print(f'  Final entropy               : {np.mean(logs["entropy"][-10:]):.3f}')
        print(f'  Final grad norm             : {np.mean(logs["grad_norm"][-10:]):.3f}')

summarize_logs(logs_baseline, 'BASELINE')

---
## Section 4: GAE λ Ablation — The Bias-Variance Tradeoff

**Research Question:** How does λ affect:
1. Sample efficiency
2. Training stability
3. Value function quality (explained variance)
4. Policy gradient variance

**Hypothesis:**
- λ=0.0 (TD): Low variance, high bias if V is inaccurate early in training → slow initial learning
- λ=1.0 (MC): Unbiased but high variance → noisy, slower convergence
- λ=0.95: Sweet spot

In [ ]:
lambda_values = [0.0, 0.5, 0.8, 0.95, 1.0]
lambda_results = {}

print('GAE LAMBDA ABLATION')
print('=' * 50)

for lam in lambda_values:
    print(f'\n  λ = {lam}')
    cfg = PPOConfig(
        env_id='CartPole-v1',
        total_timesteps=200_000,
        gae_lambda=lam,
        rollout_steps=2048,
    )
    trainer = PPOTrainer(cfg)
    logs = trainer.train(verbose=False)
    lambda_results[lam] = logs
    final_return = np.mean(logs['episode_rewards'][-30:]) if logs['episode_rewards'] else 0
    ev = np.mean(logs['explained_variance'][-10:]) if logs['explained_variance'] else 0
    print(f'    Final return: {final_return:.1f}, Explained Variance: {ev:.3f}')

In [ ]:
# Plot GAE ablation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = plt.cm.viridis(np.linspace(0, 1, len(lambda_values)))

for (lam, logs), color in zip(lambda_results.items(), colors):
    rewards = logs['episode_rewards']
    if rewards:
        axes[0].plot(smooth(rewards), color=color, label=f'λ={lam}', linewidth=2)
    if logs['explained_variance']:
        axes[1].plot(logs['explained_variance'], color=color, label=f'λ={lam}', linewidth=2)
    if logs['approx_kl']:
        axes[2].plot(logs['approx_kl'], color=color, label=f'λ={lam}', linewidth=2)

axes[0].set_title('Episode Return vs λ', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Episode'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Explained Variance (Critic Quality) vs λ', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Update'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].set_title('Approx KL Divergence vs λ', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Update'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('GAE λ Ablation — CartPole-v1', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Research Observations ──
print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESEARCH OBSERVATIONS — GAE λ
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

λ=0.0 (pure TD):
  - Advantages = just delta_t = r + γV(s') - V(s)
  - Relies ENTIRELY on V(s) being accurate
  - Early in training, V is random → bootstrapped error propagates
  - Typically SLOWER to start but more stable once V is good

λ=1.0 (pure Monte Carlo):
  - Advantages = full return - V(s_t)
  - Unbiased but HIGH variance (noisy rewards far in future)
  - Policy gradient is noisy → updates are all over the place
  - Eventual performance may be similar but training is noisier

λ=0.95 (sweet spot):
  - Exponentially weighted average of n-step returns
  - Tolerant of imperfect V early, reduces variance smoothly
  - Best empirical performance across most environments

Key insight: GAE λ and γ interact.
  Effective horizon = 1/(1 - γλ)
  With γ=0.99, λ=0.95: horizon = 1/(1-0.9405) ≈ 16.8 steps
  With γ=0.99, λ=1.0:  horizon = 1/(1-0.99)   ≈ 100 steps
""")

---
## Section 5: Clip ε Ablation

**Research Question:** How does ε control the tradeoff between:
- Sample efficiency (larger ε = more aggressive updates)
- Stability (smaller ε = safer updates)

In [ ]:
clip_values = [0.05, 0.1, 0.2, 0.3, 0.5]
clip_results = {}

print('CLIP EPSILON ABLATION')
print('=' * 50)

for eps in clip_values:
    print(f'  ε = {eps}')
    cfg = PPOConfig(
        env_id='CartPole-v1',
        total_timesteps=200_000,
        clip_eps=eps,
    )
    trainer = PPOTrainer(cfg)
    logs = trainer.train(verbose=False)
    clip_results[eps] = logs
    final_return = np.mean(logs['episode_rewards'][-30:]) if logs['episode_rewards'] else 0
    clip_frac = np.mean(logs['clip_fraction'][-10:]) if logs['clip_fraction'] else 0
    print(f'    Final return: {final_return:.1f}, Clip fraction: {clip_frac:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = plt.cm.plasma(np.linspace(0, 0.9, len(clip_values)))

for (eps, logs), color in zip(clip_results.items(), colors):
    if logs['episode_rewards']:
        axes[0].plot(smooth(logs['episode_rewards']), color=color, label=f'ε={eps}', linewidth=2)
    if logs['clip_fraction']:
        axes[1].plot(logs['clip_fraction'], color=color, label=f'ε={eps}', linewidth=2)
    if logs['approx_kl']:
        axes[2].plot(logs['approx_kl'], color=color, label=f'ε={eps}', linewidth=2)

axes[0].set_title('Episode Return vs ε', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Clip Fraction vs ε', fontsize=13, fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].axhline(0.1, color='green', linestyle='--', alpha=0.5, label='healthy low')
axes[1].axhline(0.4, color='red', linestyle='--', alpha=0.5, label='too high')
axes[2].set_title('KL Divergence vs ε', fontsize=13, fontweight='bold')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Clip ε Ablation', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESEARCH OBSERVATIONS — Clip ε
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ε=0.05 (very conservative):
  - Tiny policy updates per iteration
  - Very stable but slow
  - Clip fraction near 0 → gradient rarely flows
  - Effectively wastes data: 10 epochs but each does little

ε=0.2 (default, recommended):
  - Allows ratio in [0.8, 1.2]
  - Clip fraction 10-30%: healthy zone
  - Good balance: fast enough, stable enough

ε=0.5 (aggressive):
  - Allows ratio in [0.5, 1.5] — huge policy swings
  - On simple envs might still work
  - On harder envs: policy collapse risk
  - KL blows up → IS ratio invalid → gradient meaningless

Key diagnostic: clip_fraction
  If clip_fraction → 0: ε too large OR policy barely changing (stuck)
  If clip_fraction → 0.5+: ε too small OR learning rate too high
""")

---
## Section 6: Epoch Count (K) Ablation

**Research Question:** More epochs = more data efficiency, but at what cost?

In [ ]:
epoch_values = [1, 3, 5, 10, 20, 40]
epoch_results = {}

print('EPOCH COUNT ABLATION')
print('=' * 50)

for K in epoch_values:
    print(f'  K = {K}')
    cfg = PPOConfig(
        env_id='CartPole-v1',
        total_timesteps=200_000,
        n_epochs=K,
    )
    trainer = PPOTrainer(cfg)
    logs = trainer.train(verbose=False)
    epoch_results[K] = logs
    final_return = np.mean(logs['episode_rewards'][-30:]) if logs['episode_rewards'] else 0
    final_kl = np.mean(logs['approx_kl'][-10:]) if logs['approx_kl'] else 0
    print(f'    Final return: {final_return:.1f}, Final KL: {final_kl:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = plt.cm.cool(np.linspace(0, 1, len(epoch_values)))

for (K, logs), color in zip(epoch_results.items(), colors):
    if logs['episode_rewards']:
        axes[0].plot(smooth(logs['episode_rewards']), color=color, label=f'K={K}', linewidth=2)
    if logs['approx_kl']:
        axes[1].plot(logs['approx_kl'], color=color, label=f'K={K}', linewidth=2)
    if logs['explained_variance']:
        axes[2].plot(logs['explained_variance'], color=color, label=f'K={K}', linewidth=2)

for ax in axes:
    ax.legend(); ax.grid(alpha=0.3)
axes[0].set_title('Return vs Epochs K'); axes[1].set_title('KL vs K'); axes[2].set_title('Explained Var vs K')
axes[1].axhline(0.02, color='red', ls='--', alpha=0.5)
plt.suptitle('Epoch Count Ablation', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESEARCH OBSERVATIONS — Epoch Count K
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

K=1: Standard policy gradient (no data reuse)
  - Low KL (policy barely changes per rollout)
  - Wasteful: 2048 samples → 1 gradient step

K=10 (default): Sweet spot
  - 10 passes over same data
  - Clipping prevents overfitting to the batch
  - KL stays controlled

K=40 (too many):
  - Policy drifts far from data-collection policy
  - IS ratio r_t becomes invalid approximation
  - KL spikes → gradient signal is garbage
  - Catastrophic updates can occur

Solution for K=40: add target_kl early stopping!
  With target_kl=0.01, K=40 becomes adaptive
  (stops after e.g. 8 epochs when KL threshold reached)
""")

---
## Section 7: Entropy Coefficient Ablation

**Research Question:** How much exploration do we need?

This is especially interesting for LunarLander where premature convergence is a real problem.

In [ ]:
entropy_values = [0.0, 0.001, 0.01, 0.05, 0.1]
entropy_results = {}

print('ENTROPY COEFFICIENT ABLATION')
print('=' * 50)

for ent_coef in entropy_values:
    print(f'  ent_coef = {ent_coef}')
    cfg = PPOConfig(
        env_id='CartPole-v1',
        total_timesteps=200_000,
        ent_coef=ent_coef,
    )
    trainer = PPOTrainer(cfg)
    logs = trainer.train(verbose=False)
    entropy_results[ent_coef] = logs
    final_return = np.mean(logs['episode_rewards'][-30:]) if logs['episode_rewards'] else 0
    final_ent = np.mean(logs['entropy'][-10:]) if logs['entropy'] else 0
    print(f'    Final return: {final_return:.1f}, Final entropy: {final_ent:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = plt.cm.autumn(np.linspace(0, 1, len(entropy_values)))

for (ec, logs), color in zip(entropy_results.items(), colors):
    if logs['episode_rewards']:
        axes[0].plot(smooth(logs['episode_rewards']), color=color, label=f'c₂={ec}', linewidth=2)
    if logs['entropy']:
        axes[1].plot(logs['entropy'], color=color, label=f'c₂={ec}', linewidth=2)

axes[0].set_title('Return vs Entropy Coef', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Policy Entropy over Training', fontsize=13, fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Entropy Coefficient Ablation', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESEARCH OBSERVATIONS — Entropy Coefficient
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

ent_coef=0.0 (no entropy):
  - Policy free to become deterministic immediately
  - Might converge faster on CartPole (simple env)
  - Dangerous on harder envs with local optima

ent_coef=0.01 (default):
  - Gentle pressure toward exploration
  - Entropy decays naturally as policy improves
  - Won't dominate the policy loss

ent_coef=0.1 (strong):
  - Policy stays more random for longer
  - Can HURT on simple envs (prevents committing)
  - Can HELP on envs with many local optima
  - Used in Atari games, multi-task learning

Practical advice:
  - Monitor entropy during training
  - If entropy drops to near 0 early → increase ent_coef
  - If entropy stays high and performance plateaus → decrease
  - Some papers anneal ent_coef: high early, low late
""")

---
## Section 8: PPO-Clip vs PPO-KL

PPO-KL replaces the clip with an adaptive KL penalty:

$$L^{KL}(\theta) = L^{IS}(\theta) - \beta \cdot D_{KL}(\pi_{old} \| \pi_\theta)$$

Where $\beta$ is adaptively adjusted:
- If $KL > 1.5 \cdot d_{targ}$ → $\beta \times= 2$ (increase penalty)
- If $KL < d_{targ}/1.5$ → $\beta /= 2$ (decrease penalty)

In [ ]:
class PPOKLTrainer(PPOTrainer):
    """
    PPO with adaptive KL penalty instead of clipping.
    From Section 4 of the original PPO paper.
    """
    def __init__(self, config, target_kl=0.01, beta_init=1.0):
        super().__init__(config)
        self.beta = beta_init        # adaptive KL penalty coefficient
        self.target_kl = target_kl

    def update(self, iteration, total_iters):
        if self.cfg.anneal_lr:
            frac = 1.0 - iteration / (self.cfg.total_timesteps // self.cfg.rollout_steps)
            self.optimizer.param_groups[0]['lr'] = frac * self.cfg.lr

        epoch_metrics = defaultdict(list)

        for epoch in range(self.cfg.n_epochs):
            for obs_b, act_b, old_lp_b, adv_b, ret_b in \
                    self.buffer.get_batches(self.cfg.batch_size):

                _, new_lp, entropy, new_val = self.agent.get_action_and_value(obs_b, act_b)
                new_val = new_val.squeeze()

                log_ratio = new_lp - old_lp_b
                ratio = log_ratio.exp()

                # IS policy objective (no clipping)
                policy_obj = (ratio * adv_b).mean()

                # KL penalty
                kl = compute_approx_kl(old_lp_b, new_lp)

                # PPO-KL objective: maximize (IS - beta * KL)
                policy_loss = -(policy_obj - self.beta * kl)

                value_loss = 0.5 * ((new_val - ret_b) ** 2).mean()
                entropy_loss = entropy.mean()

                loss = policy_loss + self.cfg.vf_coef * value_loss - self.cfg.ent_coef * entropy_loss

                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.agent.parameters(), self.cfg.max_grad_norm)
                self.optimizer.step()

                epoch_metrics['policy_loss'].append(policy_loss.item())
                epoch_metrics['value_loss'].append(value_loss.item())
                epoch_metrics['entropy'].append(entropy_loss.item())
                epoch_metrics['approx_kl'].append(kl)
                epoch_metrics['clip_fraction'].append(0.0)  # no clipping
                epoch_metrics['explained_variance'].append(explained_variance(new_val, ret_b))
                epoch_metrics['grad_norm'].append(compute_grad_norm(self.agent))
                epoch_metrics['beta'].append(self.beta)

        # Adaptive beta update
        mean_kl = np.mean(epoch_metrics['approx_kl'])
        if mean_kl < self.target_kl / 1.5:
            self.beta /= 2
        elif mean_kl > self.target_kl * 1.5:
            self.beta *= 2
        self.beta = np.clip(self.beta, 1e-4, 100)

        for k, v in epoch_metrics.items():
            self.logs[k].append(np.mean(v))


print('PPO-KL Trainer defined ✓')

# Compare PPO-Clip vs PPO-KL
print('\nRunning PPO-Clip vs PPO-KL comparison...')

cfg_compare = PPOConfig(env_id='CartPole-v1', total_timesteps=200_000)

trainer_clip = PPOTrainer(cfg_compare)
logs_clip = trainer_clip.train(verbose=False)
print('  PPO-Clip done')

trainer_kl = PPOKLTrainer(cfg_compare, target_kl=0.01)
logs_kl = trainer_kl.train(verbose=False)
print('  PPO-KL done')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for logs, label, color in [
    (logs_clip, 'PPO-Clip', 'steelblue'),
    (logs_kl,   'PPO-KL',  'tomato'),
]:
    if logs['episode_rewards']:
        axes[0].plot(smooth(logs['episode_rewards']), label=label, color=color, linewidth=2)
    if logs['approx_kl']:
        axes[1].plot(logs['approx_kl'], label=label, color=color, linewidth=2)

if 'beta' in logs_kl:
    axes[2].plot(logs_kl['beta'], color='tomato', linewidth=2)
    axes[2].set_title('Adaptive β (PPO-KL)', fontsize=12)
    axes[2].set_yscale('log')
    axes[2].grid(alpha=0.3)

axes[0].set_title('Return: Clip vs KL', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('KL Divergence: Clip vs KL', fontsize=13, fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].axhline(0.01, color='black', ls='--', alpha=0.4, label='target KL')

plt.suptitle('PPO-Clip vs PPO-KL', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print("""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
RESEARCH OBSERVATIONS — PPO-Clip vs PPO-KL
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PPO-Clip:
  ✅ Simpler: no β hyperparameter
  ✅ Works with shared actor-critic (gradient flows cleanly)
  ✅ Generally more robust
  ❌ Hard constraint via clipping can prevent beneficial updates
     even when IS approximation is still valid

PPO-KL:
  ✅ Adaptive β automatically adjusts to environment
  ✅ Better principled connection to TRPO
  ❌ β needs time to adapt (burns some early experience)
  ❌ On the original paper, performed worse than Clip!
  ❌ One more hyperparameter (target_kl)

Why does Clip win? 
  Clip removes the gradient entirely for out-of-range ratios.
  This is a HARD constraint that's robust to noisy advantages.
  KL penalty is a SOFT constraint — can be overwhelmed by
  large advantage estimates.
""")

---
## Section 9: Network Architecture Ablation

**Research Question:** Shared vs. separate actor-critic networks, and depth/width tradeoffs.

In [ ]:
class SeparateActorCritic(nn.Module):
    """
    Fully separate actor and critic networks.
    
    Advantage: No gradient interference between policy and value objectives.
    Disadvantage: 2x parameters, slower to train.
    """
    def __init__(self, obs_dim, n_actions, hidden_size=64):
        super().__init__()
        self.actor_net = nn.Sequential(
            layer_init(nn.Linear(obs_dim, hidden_size)), nn.Tanh(),
            layer_init(nn.Linear(hidden_size, hidden_size)), nn.Tanh(),
            layer_init(nn.Linear(hidden_size, n_actions), std=0.01)
        )
        self.critic_net = nn.Sequential(
            layer_init(nn.Linear(obs_dim, hidden_size)), nn.Tanh(),
            layer_init(nn.Linear(hidden_size, hidden_size)), nn.Tanh(),
            layer_init(nn.Linear(hidden_size, 1), std=1.0)
        )

    def get_value(self, x):
        return self.critic_net(x)

    def get_action_and_value(self, x, action=None):
        logits = self.actor_net(x)
        dist = Categorical(logits=logits)
        if action is None:
            action = dist.sample()
        return action, dist.log_prob(action), dist.entropy(), self.critic_net(x)


class DeepActorCritic(ActorCritic):
    """Deeper network: 3 hidden layers."""
    def __init__(self, obs_dim, n_actions, hidden_size=64):
        super().__init__(obs_dim, n_actions, hidden_size)
        self.shared = nn.Sequential(
            layer_init(nn.Linear(obs_dim, hidden_size)), nn.Tanh(),
            layer_init(nn.Linear(hidden_size, hidden_size)), nn.Tanh(),
            layer_init(nn.Linear(hidden_size, hidden_size)), nn.Tanh(),
        )


# Test architectures
arch_configs = [
    ('Shared-64',    ActorCritic,        64),
    ('Shared-256',   ActorCritic,       256),
    ('Separate-64',  SeparateActorCritic, 64),
    ('Deep-64',      DeepActorCritic,    64),
]

for name, cls, h in arch_configs:
    net = cls(obs_dim=4, n_actions=2, hidden_size=h)
    n_params = sum(p.numel() for p in net.parameters())
    print(f'  {name:20s}: {n_params:,} parameters')

In [ ]:
# Monkey-patch PPOTrainer to accept custom architecture
class PPOTrainerArch(PPOTrainer):
    def __init__(self, config, arch_class, hidden_size):
        self.cfg = config
        self.env = make_env(config.env_id)
        obs_dim = self.env.observation_space.shape[0]
        n_actions = self.env.action_space.n
        self.agent = arch_class(obs_dim, n_actions, hidden_size).to(device)
        self.optimizer = optim.Adam(self.agent.parameters(), lr=config.lr, eps=1e-5)
        self.buffer = RolloutBuffer(config.rollout_steps, obs_dim, device)
        self.logs = defaultdict(list)
        self.global_step = 0

arch_results = {}
print('ARCHITECTURE ABLATION')
for name, arch_cls, h in arch_configs:
    print(f'  {name}')
    cfg = PPOConfig(env_id='CartPole-v1', total_timesteps=200_000, hidden_size=h)
    trainer = PPOTrainerArch(cfg, arch_cls, h)
    logs = trainer.train(verbose=False)
    arch_results[name] = logs
    final_return = np.mean(logs['episode_rewards'][-30:]) if logs['episode_rewards'] else 0
    print(f'    Final return: {final_return:.1f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'tomato', 'mediumseagreen', 'purple']

for (name, logs), color in zip(arch_results.items(), colors):
    if logs['episode_rewards']:
        axes[0].plot(smooth(logs['episode_rewards']), label=name, color=color, linewidth=2)
    if logs['explained_variance']:
        axes[1].plot(logs['explained_variance'], label=name, color=color, linewidth=2)

axes[0].set_title('Return by Architecture', fontsize=13, fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].set_title('Explained Variance by Architecture', fontsize=13, fontweight='bold')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('Architecture Ablation', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Section 10: LunarLander-v2 — Full Training

LunarLander is significantly harder:
- Reward shaping (shaped reward for smooth landing)
- Episode can last ~1000 steps
- Requires coordinated leg and engine control
- Prone to local optima (hovering)

In [ ]:
print('=' * 60)
print('PPO on LunarLander-v2')
print('=' * 60)

cfg_lunar = PPOConfig(
    env_id='LunarLander-v2',
    total_timesteps=1_000_000,
    rollout_steps=2048,
    gamma=0.99,
    gae_lambda=0.95,
    n_epochs=4,          # fewer epochs (LunarLander is harder, more conservative)
    batch_size=64,
    clip_eps=0.2,
    vf_coef=0.5,
    ent_coef=0.01,
    lr=3e-4,
    max_grad_norm=0.5,
    hidden_size=64,
    anneal_lr=True,
)

trainer_lunar = PPOTrainer(cfg_lunar)
logs_lunar = trainer_lunar.train(verbose=True)

plot_training(logs_lunar, title='PPO — LunarLander-v2')
summarize_logs(logs_lunar, 'LunarLander-v2')

---
## Section 11: LR Annealing Ablation

In [ ]:
lr_results = {}
print('LEARNING RATE ABLATION')

for lr, anneal in [(3e-4, True), (3e-4, False), (1e-3, True), (1e-4, True)]:
    label = f'lr={lr:.0e},anneal={anneal}'
    print(f'  {label}')
    cfg = PPOConfig(
        env_id='CartPole-v1',
        total_timesteps=200_000,
        lr=lr,
        anneal_lr=anneal,
    )
    trainer = PPOTrainer(cfg)
    logs = trainer.train(verbose=False)
    lr_results[label] = logs
    final_return = np.mean(logs['episode_rewards'][-30:]) if logs['episode_rewards'] else 0
    print(f'    Final return: {final_return:.1f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['steelblue', 'tomato', 'green', 'purple']
for (label, logs), color in zip(lr_results.items(), colors):
    if logs['episode_rewards']:
        axes[0].plot(smooth(logs['episode_rewards']), label=label, color=color, linewidth=2)
    if logs['policy_loss']:
        axes[1].plot(logs['policy_loss'], label=label, color=color, linewidth=2)
axes[0].set_title('Return by LR', fontsize=12); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].set_title('Policy Loss by LR', fontsize=12); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.suptitle('LR & Annealing Ablation', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

---
## Section 12: Comprehensive Ablation Summary

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║           COMPLETE PPO HYPERPARAMETER GUIDE                             ║
║           Based on Paper + Empirical Ablations                         ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  HYPERPARAMETER   DEFAULT    RANGE         EFFECT                        ║
║  ─────────────────────────────────────────────────────────────────────  ║
║  clip_eps         0.2       [0.1, 0.3]     ↑ = more aggressive updates  ║
║                                            KL monitor: keep <0.05       ║
║                                                                          ║
║  gae_lambda       0.95      [0.8, 1.0]     ↑ = more Monte Carlo         ║
║                                            ↓ = more bootstrapped TD     ║
║                                            Sensitive to V quality       ║
║                                                                          ║
║  n_epochs         10        [3, 30]        ↑ = more data reuse          ║
║                                            Use target_kl for safety     ║
║                                                                          ║
║  gamma            0.99      [0.95, 0.999]  ↓ = myopic agent             ║
║                                            Long horizon → 0.999         ║
║                                                                          ║
║  ent_coef         0.01      [0.0, 0.1]     ↑ = more exploration         ║
║                                            Hard envs → 0.05-0.1        ║
║                                                                          ║
║  vf_coef          0.5       [0.25, 1.0]    Critic training rate         ║
║                                            Separate net → 1.0 ok        ║
║                                                                          ║
║  lr               3e-4      [1e-5, 1e-3]   ALWAYS use anneal_lr=True    ║
║                                            Prevents late-stage drift    ║
║                                                                          ║
║  rollout_steps    2048      [128, 8192]    ↑ = better GAE estimates     ║
║                                            ↓ = more frequent updates    ║
║                                                                          ║
║  max_grad_norm    0.5       [0.3, 1.0]     NEVER remove this!           ║
║                                            Prevents gradient explosion  ║
║                                                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║  KEY DIAGNOSTIC METRICS                                                  ║
║  ─────────────────────────────────────────────────────────────────────  ║
║  explained_variance  > 0.8  → critic is good                            ║
║                       < 0.0 → critic is misleading GAE badly            ║
║                                                                          ║
║  approx_kl           < 0.02 → policy update is safe                     ║
║                       > 0.05 → reduce lr, reduce n_epochs               ║
║                                                                          ║
║  clip_fraction       0.1-0.3 → healthy                                  ║
║                       < 0.05 → epsilon too large / stuck                ║
║                       > 0.5  → epsilon too small / lr too high          ║
║                                                                          ║
║  entropy             monitor for premature collapse → 0                 ║
║                       if drops fast → increase ent_coef                 ║
║                                                                          ║
║  grad_norm           < 1.0 usually healthy                              ║
║                       spikes → KL about to blow up                      ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

---
## Section 13: The PPO Sanity Checklist

Before declaring your PPO implementation correct, verify these:

In [ ]:
def ppo_sanity_check(trainer):
    """
    Run a series of diagnostic checks on a trained PPO agent.
    Based on: https://iclr-blog-track.github.io/2022/03/25/ppo-implementation-details/
    (The famous 37 implementation details of PPO paper)
    """
    logs = trainer.logs
    agent = trainer.agent
    checks = []

    # 1. Is the value function learning? (EV should rise over time)
    if logs['explained_variance']:
        ev = logs['explained_variance']
        ev_trend = np.mean(ev[-10:]) - np.mean(ev[:10])
        ok = ev_trend > 0
        checks.append(('Value fn learning (EV increasing)', ok,
                       f'EV start={np.mean(ev[:10]):.2f} → end={np.mean(ev[-10:]):.2f}'))

    # 2. Is entropy decreasing? (policy getting more decisive)
    if logs['entropy']:
        ent = logs['entropy']
        ok = np.mean(ent[-10:]) < np.mean(ent[:10])
        checks.append(('Entropy decreasing (policy converging)', ok,
                       f'H start={np.mean(ent[:10]):.3f} → end={np.mean(ent[-10:]):.3f}'))

    # 3. Is KL divergence controlled?
    if logs['approx_kl']:
        max_kl = max(logs['approx_kl'])
        ok = max_kl < 0.1
        checks.append(('KL divergence controlled (<0.1)', ok,
                       f'Max KL observed: {max_kl:.4f}'))

    # 4. Grad norm is stable (not exploding)
    if logs['grad_norm']:
        max_gn = max(logs['grad_norm'])
        ok = max_gn < 10.0
        checks.append(('Gradient norm stable (<10)', ok,
                       f'Max grad norm: {max_gn:.3f}'))

    # 5. Policy is improving
    if logs['episode_rewards'] and len(logs['episode_rewards']) > 20:
        early = np.mean(logs['episode_rewards'][:20])
        late  = np.mean(logs['episode_rewards'][-20:])
        ok = late > early
        checks.append(('Policy is improving', ok,
                       f'Early={early:.1f} → Late={late:.1f}'))

    # 6. Orthogonal init check (initial action probabilities should be near-uniform)
    import torch
    with torch.no_grad():
        dummy_obs = torch.zeros(1, trainer.env.observation_space.shape[0]).to(device)
        _, lp, _, _ = agent.get_action_and_value(dummy_obs)
        # log_prob of uniform over n_actions should be around -log(n_actions)
        n_a = trainer.env.action_space.n
        expected_lp = -np.log(n_a)

    # Print results
    print('\n── PPO SANITY CHECKS ────────────────────────────────────────')
    all_ok = True
    for name, ok, detail in checks:
        status = '✅' if ok else '❌'
        all_ok = all_ok and ok
        print(f'  {status}  {name}')
        print(f'       {detail}')

    print(f'\n  Overall: {"All checks passed! 🎉" if all_ok else "Some checks failed — investigate above."}')


ppo_sanity_check(trainer_baseline)

---
## Section 14: Visualize What the Policy Learned

In [ ]:
# Visualize policy decisions in CartPole state space

agent = trainer_baseline.agent
agent.eval()

# CartPole obs: [cart_pos, cart_vel, pole_angle, pole_vel]
# Vary pole_angle vs pole_vel and see what the policy does

angles = np.linspace(-0.3, 0.3, 50)
ang_vels = np.linspace(-2.0, 2.0, 50)
AA, VV = np.meshgrid(angles, ang_vels)

probs_right = np.zeros_like(AA)

with torch.no_grad():
    for i in range(len(angles)):
        for j in range(len(ang_vels)):
            obs = torch.FloatTensor([[0, 0, angles[i], ang_vels[j]]]).to(device)
            logits = agent.actor(agent.shared(obs))
            probs = torch.softmax(logits, dim=-1)
            probs_right[j, i] = probs[0, 1].item()  # prob of pushing right

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im = axes[0].contourf(AA, VV, probs_right, levels=50, cmap='RdBu_r')
axes[0].contour(AA, VV, probs_right, levels=[0.5], colors='black', linewidths=2)
plt.colorbar(im, ax=axes[0], label='P(push right)')
axes[0].set_xlabel('Pole Angle (rad)'); axes[0].set_ylabel('Pole Angular Velocity')
axes[0].set_title('Policy Decision Boundary\n(black line = 50/50)', fontsize=12, fontweight='bold')
axes[0].axvline(0, color='white', ls='--', alpha=0.5)
axes[0].axhline(0, color='white', ls='--', alpha=0.5)

# Value function surface
values = np.zeros_like(AA)
with torch.no_grad():
    for i in range(len(angles)):
        for j in range(len(ang_vels)):
            obs = torch.FloatTensor([[0, 0, angles[i], ang_vels[j]]]).to(device)
            values[j, i] = agent.get_value(obs).item()

im2 = axes[1].contourf(AA, VV, values, levels=50, cmap='viridis')
plt.colorbar(im2, ax=axes[1], label='V(s)')
axes[1].set_xlabel('Pole Angle (rad)'); axes[1].set_ylabel('Pole Angular Velocity')
axes[1].set_title('Value Function V(s)\n(brighter = higher value)', fontsize=12, fontweight='bold')

plt.suptitle('What the Trained PPO Agent Learned — CartPole', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print("""
Interpretation:
  LEFT PLOT (Policy):
    Red  = push RIGHT  (high probability)
    Blue = push LEFT   (high probability)
    The policy should push RIGHT when pole tilts right (+angle)
    and also when it's falling right (positive ang_vel)
    This is physically correct! The agent learned to anticipate.
    
  RIGHT PLOT (Value Function):
    Bright = high value states (pole near vertical, slow velocity)
    Dark   = low value states (pole far tilted, falling fast)
    Center should be brightest — pole straight up is the best state.
""")

---
## Section 15: Final Research Notes & Further Reading

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║            FURTHER RESEARCH DIRECTIONS                                   ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                          ║
║  1. VALUE CLIPPING (OpenAI's trick)                                      ║
║     Also clip the value function update:                                 ║
║     v_clipped = v_old + clip(v_new - v_old, -ε, ε)                      ║
║     L_VF = max(MSE(v_new, ret), MSE(v_clipped, ret))                    ║
║     Prevents value function from changing too fast.                      ║
║     Controversial: some papers say it doesn't help.                      ║
║                                                                          ║
║  2. DUAL CLIP (Ye et al., 2020)                                          ║
║     For negative advantages, also clip from below:                       ║
║     Prevents over-penalization of actions that were                      ║
║     already penalized by a large IS ratio.                               ║
║     Helps in off-policy settings.                                        ║
║                                                                          ║
║  3. RECURRENT PPO (PPO-LSTM)                                             ║
║     Replace MLP with LSTM for partially observable envs.                 ║
║     Tricky: must reset hidden state at episode boundaries.               ║
║                                                                          ║
║  4. MULTI-AGENT PPO (MAPPO)                                              ║
║     Each agent uses PPO with centralized critic V(s_all).                ║
║     Centralized training, decentralized execution.                       ║
║                                                                          ║
║  5. PPO + INTRINSIC MOTIVATION                                           ║
║     Add RND (Random Network Distillation) reward bonus for exploration.  ║
║     Enables solving sparse reward environments.                          ║
║                                                                          ║
║  6. RLHF-PPO (ChatGPT style)                                             ║
║     Reward model replaces environment reward.                            ║
║     KL penalty added against reference policy (SFT model).              ║
║     r(x,y) = r_model(x,y) - β·KL(π_PPO || π_SFT)                       ║
║                                                                          ║
╠══════════════════════════════════════════════════════════════════════════╣
║  KEY PAPERS                                                              ║
║  ─────────────────────────────────────────────────────────────────────  ║
║  [1] PPO original: Schulman et al. (2017) arxiv:1707.06347              ║
║  [2] GAE paper: Schulman et al. (2016) arxiv:1506.02438                 ║
║  [3] TRPO: Schulman et al. (2015) arxiv:1502.05477                      ║
║  [4] 37 PPO details: Huang et al. (2022) ICLR blog track                ║
║  [5] Value clipping analysis: Engstrom et al. (2020) ICLR               ║
║  [6] RLHF: Ouyang et al. (2022) InstructGPT arxiv:2203.02155           ║
╚══════════════════════════════════════════════════════════════════════════╝
""")